# 실습 1. 로봇팔 기초 — 정기구학 · 역기구학 · PID 제어

**피지컬 AI 실습 시리즈 1/3** · CPU만으로 충분합니다.

이 실습에서는 가장 단순한 로봇인 **2-링크 평면 로봇팔**을 직접 코드로 만들어 봅니다.

| 단계 | 내용 | 핵심 질문 |
|---|---|---|
| 1-1 | 정기구학 (Forward Kinematics) | 관절 각도를 알면 손끝은 어디에 있나? |
| 1-2 | 작업 공간 (Workspace) | 이 팔이 닿을 수 있는 곳은 어디까지인가? |
| 1-3 | 역기구학 (Inverse Kinematics) | 손끝을 원하는 곳에 두려면 관절을 몇 도로? |
| 1-4 | 조금씩 다가가기 (선택) | 관절이 많은 로봇은 어떻게 푸나? |
| 1-5 | PID 제어 | 목표 각도까지 "어떻게" 움직이나? |
| 1-6 | 종합: 목표 지점 도달 | IK + 제어를 합치면? |

> 그래프의 라벨은 Colab 한글 폰트 문제를 피하려고 영어로 표기했습니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display
from ipywidgets import interact, FloatSlider

np.set_printoptions(precision=3, suppress=True)

# 로봇팔 파라미터 (단위: m)
L1, L2 = 1.0, 0.8


def show_animation(fig, update, frames, fps=25):
    """update(i)로 그림을 한 장씩 바꿔 영상으로 보여 줍니다."""
    ani = animation.FuncAnimation(fig, update, frames=frames, interval=1000 / fps)
    plt.close(fig)
    try:
        html = ani.to_html5_video()   # Colab에는 ffmpeg가 있어 가벼운 mp4로 만듦
    except Exception:
        html = ani.to_jshtml()        # ffmpeg가 없으면 프레임 묶음으로 대신 보여 줌
    display(HTML(html))

## 1-1. 정기구학 (Forward Kinematics)

어깨 각도와 팔꿈치 각도를 알면 손끝 위치가 바로 나옵니다.

1. 어깨에서 어깨 각도 방향으로 **첫 번째 팔 길이만큼** 가면 → 팔꿈치
2. 팔꿈치에서 한 번 더 꺾어 **두 번째 팔 길이만큼** 가면 → 손끝

계산은 아래 `forward_kinematics` 함수가 대신 해 줍니다. 각도만 넣으면 됩니다.
(비유: 눈을 감아도 내 손이 어디 있는지 아는 것과 같습니다.)

In [ ]:
def forward_kinematics(q, l1=L1, l2=L2):
    """관절 각도 q=(th1, th2) -> [어깨, 팔꿈치, 손끝] 좌표 (3x2 배열)"""
    th1, th2 = q
    shoulder = np.array([0.0, 0.0])
    elbow = shoulder + l1 * np.array([np.cos(th1), np.sin(th1)])
    hand = elbow + l2 * np.array([np.cos(th1 + th2), np.sin(th1 + th2)])
    return np.stack([shoulder, elbow, hand])


def draw_arm(ax, q, color="tab:blue", label=None, alpha=1.0):
    pts = forward_kinematics(q)
    ax.plot(pts[:, 0], pts[:, 1], "-o", color=color, lw=3, ms=8, alpha=alpha, label=label)


fig, ax = plt.subplots(figsize=(5, 5))
for q, c in [((0.0, 0.0), "tab:blue"), ((np.pi / 4, np.pi / 4), "tab:orange"), ((np.pi / 2, -np.pi / 2), "tab:green")]:
    draw_arm(ax, q, color=c, label=f"q=({np.degrees(q[0]):.0f}, {np.degrees(q[1]):.0f}) deg")
    print(f"q(deg)={np.degrees(q)} -> hand={forward_kinematics(q)[-1]}")
ax.set_aspect("equal"); ax.grid(alpha=0.3); ax.legend(); ax.set_title("Forward kinematics")
ax.set_xlim(-2, 2); ax.set_ylim(-2, 2)
plt.show()

### 🕹️ 슬라이더로 팔 움직여 보기
슬라이더를 움직이면 팔이 바로 따라 움직입니다. 각도를 바꿀 때 손끝 좌표(그림 제목)가 어떻게 바뀌는지 보세요.
회색 점선 원은 팔을 쭉 폈을 때 손끝이 닿는 가장 먼 거리입니다.

In [ ]:
def play_fk(shoulder_deg=30.0, elbow_deg=45.0):
    q = np.radians([shoulder_deg, elbow_deg])
    hand = forward_kinematics(q)[-1]
    fig, ax = plt.subplots(figsize=(4.5, 4.5))
    ax.add_patch(plt.Circle((0, 0), L1 + L2, fill=False, ls=":", color="gray"))
    draw_arm(ax, q)
    ax.set_title(f"hand = ({hand[0]:.2f}, {hand[1]:.2f}) m")
    ax.set_aspect("equal"); ax.grid(alpha=0.3); ax.set_xlim(-2, 2); ax.set_ylim(-2, 2)
    plt.show()


interact(play_fk,
         shoulder_deg=FloatSlider(min=-180, max=180, step=5, value=30, description="shoulder"),
         elbow_deg=FloatSlider(min=-180, max=180, step=5, value=45, description="elbow"));

## 1-2. 작업 공간 (Workspace)

관절 각도를 무작위로 많이 뽑아 손끝 위치를 찍어 보면, 로봇이 **닿을 수 있는 영역**이 보입니다.

**질문:** 링 모양의 안쪽 반지름과 바깥 반지름은 각각 얼마일까요? (힌트: 팔을 쭉 폈을 때와 최대한 접었을 때를 떠올려 보세요.)

In [ ]:
rng = np.random.default_rng(0)
qs = rng.uniform(-np.pi, np.pi, size=(5000, 2))
hands = np.array([forward_kinematics(q)[-1] for q in qs])

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(hands[:, 0], hands[:, 1], s=1, alpha=0.4)
for r in (abs(L1 - L2), L1 + L2):
    ax.add_patch(plt.Circle((0, 0), r, fill=False, ls="--", color="red"))
ax.set_aspect("equal"); ax.grid(alpha=0.3); ax.set_title("Reachable workspace")
plt.show()
print("inner radius =", abs(L1 - L2), " outer radius =", L1 + L2)

## 1-3. 역기구학 (Inverse Kinematics) — 거꾸로 풀기

이번엔 반대로, **손끝을 놓고 싶은 위치**를 주면 관절 각도를 구합니다. 계산은 `inverse_kinematics` 함수가 해 줍니다.

중요한 점은 **답이 두 개**라는 것입니다. 자기 팔로 해 보세요: 손끝을 책상 위 한 점에 둔 채로 팔꿈치를 위로 들 수도, 아래로 내릴 수도 있죠.
너무 먼 곳은 아무리 뻗어도 닿지 않으니 답이 없습니다. 실제 로봇에서는 "둘 중 어느 쪽을 고를지"도 설계 문제입니다.

In [ ]:
def inverse_kinematics(target, elbow_up=True, l1=L1, l2=L2):
    x, y = target
    c2 = (x**2 + y**2 - l1**2 - l2**2) / (2 * l1 * l2)
    if abs(c2) > 1:
        raise ValueError(f"target {target} is out of reach")
    s2 = np.sqrt(1 - c2**2) * (-1 if elbow_up else 1)
    th2 = np.arctan2(s2, c2)
    th1 = np.arctan2(y, x) - np.arctan2(l2 * s2, l1 + l2 * c2)
    return np.array([th1, th2])


target = np.array([1.2, 0.6])
fig, ax = plt.subplots(figsize=(5, 5))
for up, c in [(True, "tab:orange"), (False, "tab:green")]:
    q = inverse_kinematics(target, elbow_up=up)
    err = np.linalg.norm(forward_kinematics(q)[-1] - target)
    print(f"elbow_up={up}: q(deg)={np.degrees(q)}, FK error={err:.2e}")
    draw_arm(ax, q, color=c, label=f"elbow_up={up}")
ax.plot(*target, "r*", ms=18, label="target")
ax.set_aspect("equal"); ax.grid(alpha=0.3); ax.legend(); ax.set_title("Two IK solutions")
ax.set_xlim(-0.5, 2); ax.set_ylim(-0.5, 2)
plt.show()

### 🕹️ 슬라이더로 손끝 목표 옮겨 보기
이번에는 **손끝 위치(빨간 별)** 를 슬라이더로 옮깁니다. 관절 각도는 역기구학이 계산합니다.
팔꿈치를 든 답(주황)과 내린 답(초록)이 함께 그려집니다. 별을 회색 원 밖으로 보내면 어떻게 되나요?

In [ ]:
def play_ik(x=1.2, y=0.6):
    target = np.array([x, y])
    fig, ax = plt.subplots(figsize=(4.5, 4.5))
    ax.add_patch(plt.Circle((0, 0), L1 + L2, fill=False, ls=":", color="gray"))
    try:
        for up, c in [(True, "tab:orange"), (False, "tab:green")]:
            q = inverse_kinematics(target, elbow_up=up)
            draw_arm(ax, q, color=c, label=f"elbow_up={up}: ({np.degrees(q[0]):.0f}, {np.degrees(q[1]):.0f}) deg")
        ax.legend(loc="lower left", fontsize=8)
        ax.set_title("two solutions")
    except ValueError:
        ax.set_title("out of reach: no solution", color="red")
    ax.plot(x, y, "r*", ms=18)
    ax.set_aspect("equal"); ax.grid(alpha=0.3); ax.set_xlim(-2, 2); ax.set_ylim(-2, 2)
    plt.show()


interact(play_ik,
         x=FloatSlider(min=-2, max=2, step=0.05, value=1.2),
         y=FloatSlider(min=-2, max=2, step=0.05, value=0.6));

## 1-4. 조금씩 다가가기 (선택)

관절이 7개 이상이거나 구조가 복잡한 로봇팔은 공식 하나로 답을 구하기 어렵습니다. 그래서 컴퓨터가
**"관절을 살짝 돌려 본다 → 손끝이 목표에 가까워졌는지 확인한다 → 또 살짝 돌린다"** 를 반복합니다.

아래 그림에서 어두운 색에서 밝은 색으로 갈수록 팔이 목표(빨간 별)에 다가갑니다. 코드 속 수학은 몰라도 괜찮습니다.
몇 번 만에 도착하는지만 확인해 보세요.

In [ ]:
def jacobian(q, l1=L1, l2=L2):
    th1, th2 = q
    s1, c1 = np.sin(th1), np.cos(th1)
    s12, c12 = np.sin(th1 + th2), np.cos(th1 + th2)
    return np.array([[-l1 * s1 - l2 * s12, -l2 * s12],
                     [ l1 * c1 + l2 * c12,  l2 * c12]])


def numerical_ik(target, q0, lam=0.05, steps=100, tol=1e-4):
    q = np.array(q0, dtype=float)
    history = [q.copy()]
    for _ in range(steps):
        e = target - forward_kinematics(q)[-1]
        if np.linalg.norm(e) < tol:
            break
        J = jacobian(q)
        dq = J.T @ np.linalg.solve(J @ J.T + lam**2 * np.eye(2), e)
        q += dq
        history.append(q.copy())
    return q, np.array(history)


q_sol, hist = numerical_ik(target, q0=[0.1, 0.1])
print(f"converged in {len(hist) - 1} iterations, q(deg)={np.degrees(q_sol)}")

fig, ax = plt.subplots(figsize=(5, 5))
for i, q in enumerate(hist):
    draw_arm(ax, q, color=plt.cm.viridis(i / max(len(hist) - 1, 1)), alpha=0.6)
ax.plot(*target, "r*", ms=18)
ax.set_aspect("equal"); ax.grid(alpha=0.3); ax.set_title("Numerical IK iterations (dark -> bright)")
plt.show()

## 1-5. PID 제어 — "목표 각도까지 어떻게 갈 것인가"

IK는 **어디로** 갈지를 알려 줄 뿐, **어떻게** 갈지는 알려 주지 않습니다. 모터에 얼마의 토크를 줄지 정하는 것이 제어기입니다.

중력을 받는 관절 하나(팔을 옆으로 들어 올리는 어깨를 떠올리세요)를 90°까지 들어 올려 봅니다.

**모터 힘 = P(지금 오차) + I(그동안 쌓인 오차) + D(오차가 변하는 속도)** · 오차 = 목표 각도 − 현재 각도

- **P는 용수철**: 목표에서 멀수록 세게 당김
- **D는 충격 흡수기**: 빨리 움직일수록 브레이크
- **I는 끈기**: 남은 오차가 계속 쌓이면 점점 더 세게 밀어 끝까지 도달

코드의 `Kp`, `Ki`, `Kd`는 각각을 얼마나 세게 쓸지 정하는 손잡이(게인)입니다.

아래 세 가지를 비교해 보세요.
- **P만**: 출렁이며 진동합니다.
- **PD**: 진동은 잡히지만, 중력이 계속 끌어내려서 목표보다 아래에서 멈춥니다.
- **PID**: 적분항이 남은 오차를 누적해 없앱니다.

In [ ]:
def simulate_joint(Kp, Ki, Kd, target=np.pi / 2, T=4.0, dt=0.001, m=1.0, l=0.5, b=0.1, g=9.81, tau_max=20.0):
    I = m * l**2
    th, dth, integ = 0.0, 0.0, 0.0
    prev_e = target - th
    ts, ths = [], []
    for k in range(int(T / dt)):
        e = target - th
        integ += e * dt
        de = (e - prev_e) / dt
        prev_e = e
        tau = np.clip(Kp * e + Ki * integ + Kd * de, -tau_max, tau_max)
        ddth = (tau - b * dth - m * g * l * np.sin(th)) / I
        dth += ddth * dt
        th += dth * dt
        ts.append(k * dt); ths.append(th)
    return np.array(ts), np.array(ths)


gains = {"P   (Kp=10)": (10, 0, 0),
         "PD  (Kp=10, Kd=1)": (10, 0, 1.0),
         "PID (Kp=10, Ki=15, Kd=1)": (10, 15, 1.0)}

fig, ax = plt.subplots(figsize=(8, 4))
for name, (kp, ki, kd) in gains.items():
    t, th = simulate_joint(kp, ki, kd)
    ax.plot(t, np.degrees(th), label=name)
    last = np.degrees(th[t > t[-1] - 1.0])  # 마지막 1초 구간
    print(f"{name:28s} last-1s mean = {last.mean():6.2f} deg, swing = ±{(last.max() - last.min()) / 2:5.2f} deg (target 90)")
ax.axhline(90, color="k", ls="--", lw=1, label="target")
ax.set_xlabel("time [s]"); ax.set_ylabel("joint angle [deg]"); ax.grid(alpha=0.3); ax.legend()
ax.set_title("P vs PD vs PID step response")
plt.show()

### 🎬 세 제어기로 팔을 들어 올리는 모습
위 그래프를 실제 움직임으로 봅니다. 아래로 늘어진 팔을 **수평(점선, 90°)** 까지 들어 올립니다. 실제 시간과 같은 속도입니다.
- **P**: 점선을 지나쳤다 돌아오기를 반복합니다.
- **PD**: 흔들림은 멈추지만 점선보다 아래에 머뭅니다. 중력이 끌어내리기 때문입니다.
- **PID**: 점선까지 올라가 멈춥니다.

In [ ]:
runs = {name.split()[0]: simulate_joint(*g) for name, g in gains.items()}
SKIP = 40                                  # 1ms마다 계산한 결과에서 40개마다 한 장 → 1초에 25장
n_frames = len(next(iter(runs.values()))[0]) // SKIP

fig, axes = plt.subplots(1, 3, figsize=(10, 3.8))
panels = []
for ax, (name, (ts, ths)) in zip(axes, runs.items()):
    ax.plot([0, 0.6], [0, 0], "k--", lw=1)                         # 목표: 수평
    ax.add_patch(plt.Circle((0, 0), 0.5, fill=False, ls=":", color="lightgray"))
    trail, = ax.plot([], [], color="tab:orange", lw=1, alpha=0.6)  # 손끝 궤적
    link, = ax.plot([], [], "-", lw=8, color="tab:blue", solid_capstyle="round")
    ax.plot(0, 0, "ko", ms=10)
    label = ax.text(0, -0.68, "", ha="center", fontsize=11)
    ax.set_title(name, fontsize=14, fontweight="bold")
    ax.set_xlim(-0.6, 0.7); ax.set_ylim(-0.75, 0.6); ax.set_aspect("equal"); ax.axis("off")
    panels.append((link, trail, label, ts, ths))


def update(i):
    k = i * SKIP
    for link, trail, label, ts, ths in panels:
        tip = 0.5 * np.array([np.sin(ths[:k + 1]), -np.cos(ths[:k + 1])])
        link.set_data([0, tip[0, -1]], [0, tip[1, -1]])
        trail.set_data(tip[0, -60 * SKIP::SKIP], tip[1, -60 * SKIP::SKIP])
        label.set_text(f"t = {ts[k]:.1f} s   angle = {np.degrees(ths[k]):5.1f} deg")


show_animation(fig, update, n_frames)

### 🕹️ 직접 해 보기: 게인 슬라이더
슬라이더를 놓으면 그래프가 다시 그려집니다. 점선(90°)에 빨리, 흔들림 없이 도달하는 조합을 찾아보세요.
1. `Kp`를 50으로 올려 보세요. 빨라지지만 무엇이 나빠지나요?
2. `Ki`를 50으로 올리면 어떤 현상(목표를 지나쳤다 돌아오는 오버슈트)이 생기나요?
3. 좋은 조합을 찾은 뒤 `mass`(팔 무게)를 2배로 바꿔도 잘 동작하나요? → **로봇이 바뀌면 튜닝도 다시 해야 한다**는 점이 학습 기반 제어(실습 2)로 넘어가는 동기입니다.

In [ ]:
def try_gains(Kp=10.0, Ki=15.0, Kd=1.0, mass=1.0):
    t, th = simulate_joint(Kp, Ki, Kd, m=mass)
    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.plot(t, np.degrees(th), lw=2)
    ax.axhline(90, color="k", ls="--", lw=1)
    ax.set_ylim(-20, 200); ax.set_xlabel("time [s]"); ax.set_ylabel("joint angle [deg]"); ax.grid(alpha=0.3)
    ax.set_title(f"Kp={Kp:.0f}, Ki={Ki:.0f}, Kd={Kd:.1f}, mass={mass:.1f}  ->  final {np.degrees(th[-1]):.1f} deg")
    plt.show()


interact(try_gains,
         Kp=FloatSlider(min=0, max=60, step=1, value=10, continuous_update=False),
         Ki=FloatSlider(min=0, max=60, step=1, value=15, continuous_update=False),
         Kd=FloatSlider(min=0, max=5, step=0.1, value=1.0, continuous_update=False),
         mass=FloatSlider(min=0.5, max=3, step=0.1, value=1.0, continuous_update=False));

## 1-6. 종합: IK로 목표를 정하고, PD로 따라가기

손끝을 여러 목표 지점에 차례로 보내 봅니다. IK로 목표 각도를 정하고, 관절마다 PD 제어로 부드럽게 따라가게 합니다.

In [ ]:
def reach_sequence(targets, Kp=40.0, Kd=12.0, dt=0.01, steps_per_target=150):
    q = np.array([0.0, 0.0]); dq = np.zeros(2)
    hand_path, q_path = [], []
    for tgt in targets:
        q_goal = inverse_kinematics(tgt, elbow_up=True)
        for _ in range(steps_per_target):
            # 각도 차이를 -pi~pi로 감싸서 먼 길로 돌지 않게 함
            e = (q_goal - q + np.pi) % (2 * np.pi) - np.pi
            u = Kp * e - Kd * dq
            dq += u * dt
            q += dq * dt
            hand_path.append(forward_kinematics(q)[-1])
            q_path.append(q.copy())
    return np.array(hand_path), q, np.array(q_path)


targets = [np.array(p) for p in [(1.2, 0.6), (0.3, 1.4), (-1.0, 0.8), (0.9, -0.9)]]
path, q_final, q_path = reach_sequence(targets)
print("final hand error to T4:", np.linalg.norm(path[-1] - targets[-1]).round(4), "m")

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(path[:, 0], path[:, 1], lw=1.5, label="hand path")
for i, t in enumerate(targets):
    ax.plot(*t, "r*", ms=15); ax.annotate(f"T{i + 1}", t + 0.05)
draw_arm(ax, q_final, color="k", alpha=0.7, label="final pose")
ax.set_aspect("equal"); ax.grid(alpha=0.3); ax.legend(); ax.set_title("IK + PD reaching")
ax.set_xlim(-2, 2); ax.set_ylim(-2, 2)
plt.show()

### 🎬 네 목표를 차례로 찾아가는 모습
지금 향하는 목표는 빨간 별, 이미 다녀온 목표는 회색 별입니다. 주황 선은 손끝이 지나간 길입니다.

In [ ]:
STEPS_PER_TARGET, SKIP = 150, 4            # 0.01초 간격 → 4개마다 한 장 = 실제 속도
fig, ax = plt.subplots(figsize=(5, 5))
stars = [ax.plot(*t, "*", ms=18, color="lightgray")[0] for t in targets]
for i, t in enumerate(targets):
    ax.annotate(f"T{i + 1}", t + 0.07)
trail, = ax.plot([], [], color="tab:orange", lw=1.5)
arm, = ax.plot([], [], "-o", lw=5, ms=9, color="tab:blue")
title = ax.set_title("")
ax.set_aspect("equal"); ax.grid(alpha=0.3); ax.set_xlim(-2, 2); ax.set_ylim(-2, 2)


def update(i):
    k = i * SKIP
    goal = min(k // STEPS_PER_TARGET, len(targets) - 1)
    for j, s in enumerate(stars):
        s.set_color("red" if j == goal else "lightgray")
    pts = forward_kinematics(q_path[k])
    arm.set_data(pts[:, 0], pts[:, 1])
    trail.set_data(path[:k + 1, 0], path[:k + 1, 1])
    title.set_text(f"going to T{goal + 1}   (t = {k * 0.01:.1f} s)")


show_animation(fig, update, len(q_path) // SKIP)

## 정리

- **정기구학**: 관절 → 손끝. 계산이 쉽고 항상 답이 하나.
- **역기구학**: 손끝 → 관절. 답이 여러 개거나 없을 수 있음. 관절이 많으면 조금씩 다가가는 반복으로 풂.
- **PID**: 목표까지의 "움직임"을 만드는 고전 제어. 모델이 바뀌면 재튜닝 필요.

고전 로보틱스는 **사람이 모델과 규칙을 설계**합니다. 다음 실습에서는 규칙 대신 **보상만 주고 로봇이 스스로 제어법을 배우게** 합니다.